# YouTube Trending Videos — Gold Layer Datamarts
### Silver → Analytics-Ready Business Views

**Purpose:** This notebook reads the cleansed silver-layer star schema and builds **analytics-ready gold datamarts**. Each table answers a specific business question without requiring joins at query time.

**Silver Tables Referenced:**

| Silver Table | Used In Gold Datamarts |
|---|---|
| `dim_category` | Category Performance |
| `dim_tags` + `bridge_video_tags` | Tag Trend Analysis |
| `dim_date_new` | Weekend vs Weekday Analysis |
| `fact_trending_videos` | Category Performance, Tag Trends, Temporal Patterns |
| `fact_video_trending_trajectory` | Trending Velocity & Peak Day Analysis |
| `fact_channel_daily_performance` | Daily Leaderboard, Consistency Scoring |

**Gold Tables Produced:**

| # | Gold Table | Description |
|---|-----------|-------------|
| 1 | `category_performance` | Category-level trending stats, engagement, and sentiment |
| 2 | `tag_trends` | Tag popularity ranked by total views & engagement |
| 3 | `trending_velocity` | Fastest-growing videos with peak day detection |
| 4 | `weekend_vs_weekday` | Temporal trending patterns by day of week |
| 5 | `channel_daily_leaderboard` | Top 10 channels per trending day |
| 6 | `channel_consistency` | Consistency score: powerhouse vs one-hit-wonder |

---
_Upstream: [silver](#) · Downstream: [gold_business_cases](#)

In [0]:
# ── PySpark imports ──
# Standard toolkit for aggregations, window rankings, and type casting.

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType, BooleanType, TimestampType

In [0]:
# ── Medallion architecture paths (ADLS Gen2) ──
# Gold layer reads from silver/ and writes analytics-ready output to gold/.

root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# ── Unity Catalog schema references ──
bronze_sch = "bronze_youtube"
silver_sch = "silver_youtube"
gold_sch = "gold_youtube"
youtube_db = "employeedatacatalog"

In [0]:
# ── Load all 9 silver tables we’ll reference across the datamarts ──

# Dimensions
dim_channel  = spark.table(f"{youtube_db}.{silver_sch}.dim_channel")           # Lifetime channel aggregates
dim_category = spark.table(f"{youtube_db}.{silver_sch}.dim_category")          # Category ID → name lookup
dim_video    = spark.table(f"{youtube_db}.{silver_sch}.dim_video")             # Video metadata
dim_tags     = spark.table(f"{youtube_db}.{silver_sch}.dim_tags")              # Unique tag lookup
dim_date     = spark.table(f"{youtube_db}.{silver_sch}.dim_date_new")          # Calendar dimension

# Bridge
bridge_vt    = spark.table(f"{youtube_db}.{silver_sch}.bridge_video_tags")     # Video ↔ tag bridge

# Facts
fact         = spark.table(f"{youtube_db}.{silver_sch}.fact_trending_videos")  # Core trending fact
fact_traj    = spark.table(f"{youtube_db}.{silver_sch}.fact_video_trending_trajectory")  # Day-over-day deltas
fact_ch_day  = spark.table(f"{youtube_db}.{silver_sch}.fact_channel_daily_performance")  # Daily channel rollup

print("Silver tables loaded successfully")

In [0]:
# ============================================================
# GOLD TABLE 1: category_performance
# ============================================================
# Rolls up trending data by YouTube category. Shows which categories
# dominate the trending page by views, engagement, and sentiment.
#
# Business question: "Which content categories get the most traction
# on YouTube trending, and how does audience sentiment differ?"

df_gold_cat = (
    fact.join(dim_category, fact["category_key"] == dim_category["category_key"], "left")
    .groupBy(col("category_name"))
    .agg(
        count("fact_key").alias("total_trending_entries"),          # Total times videos in this category trended
        countDistinct("video_id").alias("unique_videos"),
        countDistinct("channel_key").alias("unique_channels"),
        sum("views").alias("total_views"),
        round(avg("views"), 0).alias("avg_views"),
        max("views").alias("max_views"),
        sum("likes").alias("total_likes"),
        sum("dislikes").alias("total_dislikes"),
        sum("comment_count").alias("total_comments"),
        round(avg("engagement_rate"), 3).alias("avg_engagement_rate"),
        round(avg("like_ratio"), 3).alias("avg_like_ratio"),
        round(avg("days_to_trend"), 1).alias("avg_days_to_trend")
    )
    .withColumn("dislike_ratio",                                   # % of reactions that are dislikes
        round(col("total_dislikes") / nullif(col("total_likes") + col("total_dislikes"), lit(0)) * 100, 3))
    .withColumn("_gold_ingested_at", current_timestamp())
    .orderBy(desc("total_views"))
)

# Persist to gold Delta and register in UC
df_gold_cat.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/category_performance")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.category_performance
              USING DELTA LOCATION '{gold_path}/category_performance'""")

print(f"Categories analyzed: {df_gold_cat.count():,}")
df_gold_cat.limit(10).display()

In [0]:
# ============================================================
# GOLD TABLE 2: tag_trends
# ============================================================
# Which tags drive the most views and engagement on trending videos?
# Joins bridge_video_tags → dim_tags → fact_trending_videos to
# aggregate performance metrics per tag.
#
# Business question: "What should I tag my video with to maximize
# reach? Which tags correlate with higher engagement?"

# Step 1 — Join tags to fact data through the bridge table
df_tag_facts = (
    bridge_vt.alias("bvt")
    .join(dim_tags.alias("t"), col("bvt.tag_key") == col("t.tag_key"), "inner")
    .join(fact.alias("f"), col("bvt.video_key") == col("f.video_key"), "inner")
)

# Step 2 — Aggregate by tag
df_gold_tags = (
    df_tag_facts
    .groupBy(col("t.tag_name"))
    .agg(
        count("f.fact_key").alias("trending_appearances"),          # Trending-day entries featuring this tag
        countDistinct("f.video_id").alias("unique_videos"),
        countDistinct("f.channel_key").alias("unique_channels"),
        round(sum("f.views"), 0).alias("total_views"),
        round(avg("f.views"), 0).alias("avg_views_per_entry"),
        round(avg("f.engagement_rate"), 3).alias("avg_engagement_rate"),
        round(avg("f.like_ratio"), 3).alias("avg_like_ratio"),
        round(avg("f.days_to_trend"), 1).alias("avg_days_to_trend")
    )
    .withColumn("tag_popularity_rank",                             # Rank by total views
        rank().over(Window.orderBy(desc("total_views"))))
    .withColumn("_gold_ingested_at", current_timestamp())
    .orderBy("tag_popularity_rank")
)

# Persist to gold Delta and register in UC
df_gold_tags.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/tag_trends")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.tag_trends
              USING DELTA LOCATION '{gold_path}/tag_trends'""")

print(f"Tags analyzed: {df_gold_tags.count():,}")
df_gold_tags.limit(10).display()

In [0]:
# ============================================================
# GOLD TABLE 3: trending_velocity
# ============================================================
# Uses the trajectory table to identify which videos grew fastest
# and when they peaked. Computes per-video summary metrics:
#   - peak_day: the trending day with the highest daily view gain
#   - max daily views delta and total trajectory views
#   - avg growth rate across the trending window
#
# Business question: "Which videos went most viral? When do they
# typically peak — day 1, 2, 3?"

# Step 1 — Per-video trajectory summary
df_velocity = (
    fact_traj
    .filter(col("trending_day_num") > 1)                           # Only rows with a prior day to compare
    .groupBy("video_key", "video_id")
    .agg(
        count("*").alias("total_trending_days"),
        max("views").alias("peak_views"),
        max("views_delta").alias("max_daily_view_gain"),            # Biggest single-day jump
        round(avg("views_delta"), 0).alias("avg_daily_view_gain"),
        round(avg("views_growth_pct"), 2).alias("avg_daily_growth_pct"),
        round(avg("engagement_rate"), 3).alias("avg_engagement_rate"),
        round(avg("like_ratio"), 3).alias("avg_like_ratio"),
        max("trending_day_num").alias("last_trending_day")
    )
)

# Step 2 — Identify which day was the peak for each video
w_peak = Window.partitionBy("video_key").orderBy(desc("views_delta"))

df_peak_day = (
    fact_traj
    .filter(col("trending_day_num") > 1)
    .withColumn("rn", row_number().over(w_peak))
    .filter(col("rn") == 1)
    .select(
        col("video_key").alias("pk_video_key"),
        col("trending_day_num").alias("peak_day_num"),
        col("trending_date").alias("peak_date")
    )
)

# Step 3 — Join and rank by virality
df_gold_velocity = (
    df_velocity.alias("v")
    .join(df_peak_day.alias("p"), col("v.video_key") == col("p.pk_video_key"), "left")
    .withColumn("virality_rank",
        rank().over(Window.orderBy(desc("max_daily_view_gain"))))
    .withColumn("_gold_ingested_at", current_timestamp())
    .select(
        col("v.video_key"), col("v.video_id"), "total_trending_days",
        "peak_views", "max_daily_view_gain", "avg_daily_view_gain",
        "avg_daily_growth_pct", "avg_engagement_rate", "avg_like_ratio",
        "last_trending_day", "peak_day_num", "peak_date",
        "virality_rank", "_gold_ingested_at"
    )
    .orderBy("virality_rank")
)

# Persist to gold Delta and register in UC
df_gold_velocity.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/trending_velocity")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.trending_velocity
              USING DELTA LOCATION '{gold_path}/trending_velocity'""")

print(f"Videos with trajectory: {df_gold_velocity.count():,}")
df_gold_velocity.limit(10).display()

In [0]:
# ============================================================
# GOLD TABLE 4: weekend_vs_weekday
# ============================================================
# Joins fact → dim_date to analyze whether videos perform
# differently on weekends vs weekdays.
#
# Business question: "Should I publish on Monday or Saturday?
# Which day of the week gets the most trending traction?"

df_gold_temporal = (
    fact.alias("f")
    .join(dim_date.alias("d"), col("f.trending_date_key") == col("d.date_key"), "inner")
    .groupBy(
        col("d.day_name"),
        col("d.day_of_week"),
        col("d.is_weekend")
    )
    .agg(
        count("f.fact_key").alias("trending_entries"),
        countDistinct("f.video_id").alias("unique_videos"),
        round(sum("f.views"), 0).alias("total_views"),
        round(avg("f.views"), 0).alias("avg_views"),
        round(avg("f.engagement_rate"), 3).alias("avg_engagement_rate"),
        round(avg("f.like_ratio"), 3).alias("avg_like_ratio"),
        round(avg("f.days_to_trend"), 1).alias("avg_days_to_trend"),
        round(sum("f.likes"), 0).alias("total_likes"),
        round(sum("f.comment_count"), 0).alias("total_comments")
    )
    .withColumn("day_type",
        when(col("d.is_weekend") == True, lit("Weekend")).otherwise(lit("Weekday")))
    .withColumn("_gold_ingested_at", current_timestamp())
    .orderBy("d.day_of_week")
)

# Persist to gold Delta and register in UC
df_gold_temporal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/weekend_vs_weekday")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.weekend_vs_weekday
              USING DELTA LOCATION '{gold_path}/weekend_vs_weekday'""")

df_gold_temporal.display()

In [0]:
# ============================================================
# GOLD TABLE 5: channel_daily_leaderboard
# ============================================================
# Ranks channels within each trending day by total views.
# Keeps the top 10 per day for a compact leaderboard view.
#
# Business question: "Who dominated the trending page each day?
# Are the same channels always on top, or does it rotate?"

w_daily = Window.partitionBy("trending_date").orderBy(desc("daily_total_views"))

df_gold_leaderboard = (
    fact_ch_day
    .withColumn("daily_rank", rank().over(w_daily))
    .filter(col("daily_rank") <= 10)                               # Keep top 10 per day
    .withColumn("_gold_ingested_at", current_timestamp())
    .select(
        "trending_date", "daily_rank", "channel_key", "channel_title",
        "trending_video_count", "daily_total_views", "daily_total_likes",
        "daily_total_comments", "avg_engagement_rate", "avg_like_ratio",
        "top_video_views", "categories_represented", "_gold_ingested_at"
    )
    .orderBy("trending_date", "daily_rank")
)

# Persist to gold Delta and register in UC
df_gold_leaderboard.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/channel_daily_leaderboard")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.channel_daily_leaderboard
              USING DELTA LOCATION '{gold_path}/channel_daily_leaderboard'""")

print(f"Leaderboard rows (top 10/day): {df_gold_leaderboard.count():,}")
df_gold_leaderboard.limit(10).display()

In [0]:
# ============================================================
# GOLD TABLE 6: channel_consistency
# ============================================================
# Measures how consistently a channel appears on trending.
# Distinguishes steady performers ("Powerhouse") from one-hit
# wonders by computing a consistency_score and volatility_ratio.
#
# Business question: "Is this channel a reliable content machine,
# or did they just get lucky once?"

# Step 1 — Per-channel consistency metrics
df_consistency = (
    fact_ch_day
    .groupBy("channel_key", "channel_title")
    .agg(
        count("*").alias("trending_days_count"),                    # Days they appeared on trending
        round(avg("daily_total_views"), 0).alias("avg_daily_views"),
        round(stddev("daily_total_views"), 0).alias("views_stddev"),  # Volatility
        round(sum("daily_total_views"), 0).alias("lifetime_views"),
        round(avg("trending_video_count"), 1).alias("avg_videos_per_day"),
        round(avg("avg_engagement_rate"), 3).alias("avg_engagement_rate"),
        max("daily_total_views").alias("best_day_views"),
        min("daily_total_views").alias("worst_day_views")
    )
)

# Step 2 — Total possible trending days in dataset
total_days = fact_ch_day.select("trending_date").distinct().count()

# Step 3 — Derive scores and classify into tiers
df_gold_consistency = (
    df_consistency
    .withColumn("consistency_score",                               # % of all days they appeared
        round(col("trending_days_count") / lit(total_days) * 100, 2))
    .withColumn("volatility_ratio",                                # Coefficient of variation
        round(col("views_stddev") / nullif(col("avg_daily_views"), lit(0)), 3))
    .withColumn("channel_tier",
        when(col("consistency_score") >= 50, lit("Powerhouse"))        # Trending 50%+ of all days
        .when(col("consistency_score") >= 20, lit("Regular"))          # 20-50%
        .when(col("consistency_score") >= 5,  lit("Occasional"))       # 5-20%
        .otherwise(lit("One-Hit Wonder")))                             # <5%
    .withColumn("channel_tier_rank",
        rank().over(Window.orderBy(desc("consistency_score"), desc("lifetime_views"))))
    .withColumn("_gold_ingested_at", current_timestamp())
    .orderBy("channel_tier_rank")
)

# Persist to gold Delta and register in UC
df_gold_consistency.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/channel_consistency")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.channel_consistency
              USING DELTA LOCATION '{gold_path}/channel_consistency'""")

# Summary
print(f"Total trending days in dataset: {total_days}")
print(f"Channels scored: {df_gold_consistency.count():,}")
print("\nTier breakdown:")
df_gold_consistency.groupBy("channel_tier").agg(count("*").alias("channels")).orderBy(desc("channels")).display()
df_gold_consistency.limit(10).display()